# Reavaliação do EPITOME com XLM-R — JP

Este notebook treina três avaliadores (ER, IP, EX) com **dois encoders `FacebookAI/xlm-roberta-base`**, atenção entre os encoders e duas tarefas: classificação dos níveis 0/1/2 e extração de rationales. Em seguida classifica os mesmos **60 pares**, compara com a avaliação original e com a rodada anterior, e gera tabelas, testes e um relatório.

## Como rodar

1. Selecione **Ambiente de execução → Alterar tipo de ambiente de execução → GPU**. T4 é adequada para a configuração inicial.
2. Use **Executar tudo**. As entradas das rodadas anteriores são baixadas do repositório e conferidas por hash. Para enviar outros CSVs, defina `USE_PUBLISHED_BASELINES = False`. Autorize a montagem do seu Drive.
3. Aguarde ER, IP e EX. O ZIP `resultados_epitome_xlmr.zip` será baixado no final e também ficará em `Meu Drive/EPITOME_XLMR_JP/run_...`.
4. Guarde esse ZIP e devolva-o para preparar as tabelas e a resposta ao revisor.

Os pesos dos três avaliadores também ficam no Drive e ocupam aproximadamente **3,5 GB**. Eles são diferentes dos pesos do modelo de suporte a álcool e saúde. Se a sessão cair, execute este notebook de novo com a mesma configuração: tarefas concluídas serão recuperadas. Uma tarefa interrompida no meio do treino recomeça; não há retomada exata do otimizador no meio de uma época.

## Método e limites

O conjunto do projeto contém traduções PT de 3.084 pares anotados em inglês. Os rationales de parte das traduções não correspondem aos textos e alguns IDs foram alterados. Por isso:

- O treino usa **níveis EN e PT**, mas supervisiona rationales **apenas nas versões EN originais** com trechos alinháveis. Trechos ausentes/não alinháveis são ignorados na perda de rationale, sem inventar anotação.
- Divisão de aproximadamente **60% treino / 20% validação / 20% teste**, por grupo de enunciado. Duplicatas e versões EN/PT do mesmo enunciado ficam juntas.
- O checkpoint é escolhido pela **Macro-F1 na validação PT**; o teste EN/PT é reservado até o fim.
- Os níveis PT são transferidos das anotações EN. Ainda não são uma reanotação humana independente em português.

Esta é uma **adaptação do EPITOME**, com processamento atualizado e treino bilíngue. As mudanças estão no manifesto. Não deve ser descrita como reprodução numérica exata do experimento anterior ou como validação clínica. Fonte: [Sharma et al. (2020)](https://aclanthology.org/2020.emnlp-main.425/).

## Registro da execução

A configuração registrada está em `experiments/2026-10-01/manifest.json`. Esta versão inclui a correção da regressão ER. Para gerar tabelas corrigidas a partir de uma execução já concluída, use `scripts/analyze_epitome_results.py`, sem treinar novamente.


In [ ]:
import os, subprocess, sys
from pathlib import Path
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HOME'] = '/content/epitome_hf_cache'
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
print('Python:', sys.version, flush=True)
print('Instalando bibliotecas compatíveis com Python 3.13...', flush=True)
command = [
    sys.executable, '-m', 'pip', 'install', '--only-binary=:all:',
    'transformers==4.48.3', 'tokenizers==0.21.4', 'sentencepiece==0.2.2', 'numpy==2.1.3',
    'pandas==2.2.3', 'scikit-learn==1.6.1', 'scipy==1.16.3',
    'statsmodels==0.14.6', 'requests==2.32.4', 'tqdm==4.67.3', 'safetensors==0.8.0', 'packaging==26.3',
]
log_path = Path('/content/epitome_instalacao.log')
with log_path.open('w', encoding='utf-8') as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end='', flush=True)
        log.write(line)
    exit_code = process.wait()
if exit_code:
    raise RuntimeError(
        f'A instalação falhou (código {exit_code}). O motivo aparece acima e foi salvo em {log_path}. '
        'Envie as linhas com ERROR para diagnosticar; o treino ainda não começou.'
    )
print('Bibliotecas instaladas. Os pesos CUDA/PyTorch do Colab foram preservados.')


In [ ]:
from pathlib import Path
Path('/content/epitome_xlmr.py').write_text('"""Adaptação EPITOME com XLM-R. Código incorporado no notebook entregue."""\nimport copy\nimport gc\nimport hashlib\nimport itertools\nimport json\nimport math\nimport random\nimport re\nimport shutil\nimport subprocess\nimport warnings\nfrom collections import Counter\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport requests\nimport torch\nimport transformers\nimport sklearn\nfrom sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score\nfrom sklearn.model_selection import StratifiedGroupKFold\nfrom torch import nn\nfrom torch.utils.data import DataLoader, Dataset\nfrom tqdm.auto import tqdm\nfrom transformers import AutoConfig, AutoModel, AutoTokenizer, XLMRobertaModel, get_linear_schedule_with_warmup\n\nMODEL_ID = \'FacebookAI/xlm-roberta-base\'\nTASKS = (\'ER\', \'IP\', \'EX\')\nMODELS = (\'GEMMA\', \'LLAMA\', \'GPT4\')\nSTEMS = dict(ER=\'emotional-reactions\', IP=\'interpretations\', EX=\'explorations\')\nREPOS = {\n    \'en\': (\'behavioral-data/Empathy-Mental-Health\', \'0e11f98901527550885dbf253bd2af92dbcec43b\'),\n    \'pt\': (\'JPSchettino/model_analisys-\', \'ff0379ea9eaab237be50cfff2826a5cff43315e1\'),\n}\nBLOBS = {\n    \'en\': dict(ER=\'6af98f52f26daef4dd76fe9404e165419e5193bb\', IP=\'dc806b3aab4f45b36f73bc39f043d6cc430ef17b\', EX=\'a85bd077317d7413985632d0970188bf6a32f11f\'),\n    \'pt\': dict(ER=\'d53a13f9bf46b109a91ee11c652c652a7567759e\', IP=\'f1951a93abb1fa82ab5f9532328df60d5209c853\', EX=\'dab8fea5589075704bd74eaf677844862b86033b\'),\n}\n\n\ndef write_json(path, obj):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False), encoding=\'utf-8\')\n\n\ndef normalize(s):\n    return \' \'.join(str(s).strip().strip(\'"\').strip().split())\n\n\ndef seed_everything(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\ndef download_sources(directory):\n    directory = Path(directory)\n    directory.mkdir(parents=True, exist_ok=True)\n    frames, audit = {}, {}\n    for language, (repo, revision) in REPOS.items():\n        frames[language] = {}\n        for task in TASKS:\n            path = directory / f\'{language}_{task}.csv\'\n            url = f\'https://raw.githubusercontent.com/{repo}/{revision}/dataset/{STEMS[task]}-reddit.csv\'\n            if not path.exists():\n                response = requests.get(url, timeout=120)\n                response.raise_for_status()\n                path.write_bytes(response.content)\n            raw = path.read_bytes()\n            git_sha = hashlib.sha1(f\'blob {len(raw)}\\0\'.encode() + raw).hexdigest()\n            if git_sha != BLOBS[language][task]:\n                raise ValueError(f\'Arquivo {language}/{task} difere da fonte fixada. Remova {path} e execute de novo.\')\n            frame = pd.read_csv(path, dtype={\'sp_id\': str, \'rp_id\': str}).fillna(\'\')\n            required = {\'sp_id\', \'rp_id\', \'seeker_post\', \'response_post\', \'level\', \'rationales\'}\n            if not required <= set(frame):\n                raise ValueError(f\'{language}/{task}: faltam colunas {required - set(frame)}\')\n            frame[\'level\'] = pd.to_numeric(frame.level, errors=\'raise\').astype(int)\n            if set(frame.level.unique()) != {0, 1, 2} or len(frame) != 3084:\n                raise ValueError(f\'{language}/{task}: conteúdo inesperado\')\n            if (frame[[\'seeker_post\', \'response_post\']].apply(lambda c: c.str.strip()).eq(\'\').any().any()):\n                raise ValueError(f\'{language}/{task}: há textos vazios\')\n            frame[\'row_id\'] = np.arange(len(frame))\n            frames[language][task] = frame\n            audit[f\'{language}_{task}\'] = {\'url\': url, \'git_blob\': git_sha, \'sha256\': hashlib.sha256(raw).hexdigest(), \'rows\': len(frame)}\n    # As traduções preservam a ordem/níveis, mas alguns IDs foram traduzidos ou\n    # convertidos por planilha. A fonte é fixada e os desvios são registrados.\n    reference = frames[\'en\'][\'ER\']\n    for task in TASKS:\n        en, pt = frames[\'en\'][task], frames[\'pt\'][task]\n        if not en[[\'sp_id\', \'rp_id\']].equals(reference[[\'sp_id\', \'rp_id\']]):\n            raise ValueError(\'A ordem dos três arquivos originais não coincide.\')\n        matching = (en.sp_id == pt.sp_id) & (en.rp_id == pt.rp_id)\n        if not (en.level == pt.level).all() or matching.mean() < .97:\n            raise ValueError(f\'{task}: não é seguro alinhar as versões por posição.\')\n        audit[f\'alinhamento_{task}\'] = {\'method\': \'row_position_in_pinned_files\', \'labels_identical\': True,\n                                      \'id_matches\': int(matching.sum()), \'id_mismatch_row_ids\': en.loc[~matching, \'row_id\'].tolist()}\n    return frames, audit\n\n\ndef post_groups(frame):\n    """Mesmo ID OU mesmo texto de seeker fica no mesmo grupo."""\n    parent = list(range(len(frame)))\n    def root(i):\n        while parent[i] != i:\n            parent[i] = parent[parent[i]]\n            i = parent[i]\n        return i\n    seen_id, seen_text = {}, {}\n    for i, row in enumerate(frame.itertuples()):\n        text = normalize(row.seeker_post).casefold()\n        for seen, value in [(seen_id, row.sp_id), (seen_text, text)]:\n            if value in seen:\n                a, b = root(i), root(seen[value])\n                parent[max(a, b)] = min(a, b)\n            else:\n                seen[value] = i\n    return np.array([root(i) for i in range(len(frame))])\n\n\ndef grouped_split(frames, seed):\n    reference = frames[\'en\'][\'ER\']\n    groups = post_groups(reference)\n    splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)\n    fold = np.full(len(reference), -1, dtype=int)\n    for i, (_, test) in enumerate(splitter.split(reference, reference.level, groups)):\n        fold[test] = i\n    indices = {\'train\': np.flatnonzero(fold >= 2), \'validation\': np.flatnonzero(fold == 1), \'test\': np.flatnonzero(fold == 0)}\n    for a, b in itertools.combinations(indices, 2):\n        if set(groups[indices[a]]) & set(groups[indices[b]]):\n            raise AssertionError(\'Vazamento de enunciados entre divisões.\')\n    for task in TASKS:\n        for name, ids in indices.items():\n            if set(frames[\'en\'][task].iloc[ids].level.unique()) != {0, 1, 2}:\n                raise ValueError(f\'A divisão {task}/{name} não contém as três classes.\')\n    return indices, groups\n\n\ndef align_rationales(text, annotation, offsets, level):\n    """Alinha substrings originais por caracteres; não inventa spans ausentes."""\n    labels = [0 if end > start else -100 for start, end in offsets]\n    parts = [p.strip() for p in str(annotation).split(\'|\') if p.strip()]\n    if not parts:\n        return (labels, \'no_rationale_expected\') if level == 0 else ([-100] * len(offsets), \'missing_positive_rationale\')\n    spans = []\n    for part in parts:\n        # Escapa pontuação; espaço variável é permitido sem mudar o conteúdo.\n        pattern = r\'\\s+\'.join(re.escape(word) for word in part.split())\n        matches = list(re.finditer(pattern, text, flags=re.I))\n        if not matches:\n            return [-100] * len(offsets), \'unmatched_rationale\'\n        spans.extend((m.start(), m.end()) for m in matches)\n    for i, (start, end) in enumerate(offsets):\n        if end > start and any(start < right and end > left for left, right in spans):\n            labels[i] = 1\n    if not any(label == 1 for label in labels):\n        return labels, \'rationale_outside_truncation\'\n    return labels, \'aligned\'\n\n\nclass EncodedPairs(Dataset):\n    def __init__(self, frame, tokenizer, max_length, supervise_rationales=False):\n        self.frame = frame.reset_index(drop=True).copy()\n        sp = tokenizer(self.frame.seeker_post.tolist(), padding=\'max_length\', truncation=True, max_length=max_length)\n        rp = tokenizer(self.frame.response_post.tolist(), padding=\'max_length\', truncation=True, max_length=max_length,\n                       return_offsets_mapping=True)\n        self.offsets = rp.pop(\'offset_mapping\')\n        self.tensors = {\n            \'sp_ids\': torch.tensor(sp[\'input_ids\']), \'sp_mask\': torch.tensor(sp[\'attention_mask\']),\n            \'rp_ids\': torch.tensor(rp[\'input_ids\']), \'rp_mask\': torch.tensor(rp[\'attention_mask\']),\n        }\n        self.audit = Counter()\n        if \'level\' in self.frame:\n            self.tensors[\'labels\'] = torch.tensor(self.frame.level.tolist())\n            rationales = []\n            for row, offsets in zip(self.frame.itertuples(), self.offsets):\n                if supervise_rationales:\n                    labels, status = align_rationales(row.response_post, row.rationales, offsets, row.level)\n                else:\n                    labels, status = [-100] * len(offsets), \'not_supervised_for_this_language\'\n                self.audit[status] += 1\n                rationales.append(labels)\n            self.tensors[\'rationale_labels\'] = torch.tensor(rationales)\n        # Conta truncamento por texto; preserva os offsets para a atribuição.\n        for kind, text in [(\'seeker\', self.frame.seeker_post), (\'response\', self.frame.response_post)]:\n            self.audit[f\'{kind}_truncated\'] = sum(len(tokenizer.encode(t, add_special_tokens=True)) > max_length for t in text)\n\n    def __len__(self):\n        return len(self.frame)\n\n    def __getitem__(self, i):\n        return {key: values[i] for key, values in self.tensors.items()}\n\n\nclass CrossAttention(nn.Module):\n    def __init__(self, hidden_size, dropout=.1):\n        super().__init__()\n        self.q_linear = nn.Linear(hidden_size, hidden_size)\n        self.k_linear = nn.Linear(hidden_size, hidden_size)\n        self.v_linear = nn.Linear(hidden_size, hidden_size)\n        self.out = nn.Linear(hidden_size, hidden_size)\n        self.dropout = nn.Dropout(dropout)\n        self.scale = math.sqrt(hidden_size)\n\n    def forward(self, q, k, v, key_mask):\n        scores = self.q_linear(q) @ self.k_linear(k).transpose(1, 2) / self.scale\n        scores = scores.masked_fill(~key_mask[:, None, :].bool(), torch.finfo(scores.dtype).min)\n        weights = self.dropout(torch.softmax(scores, dim=-1))\n        return self.out(weights @ self.v_linear(v))\n\n\nclass EpitomeXLMR(nn.Module):\n    def __init__(self, revision=None, config=None, dropout=.1):\n        super().__init__()\n        if config is not None:  # usado nos testes com modelo pequeno e sem rede\n            self.seeker = XLMRobertaModel(config, add_pooling_layer=False)\n        else:\n            self.seeker = AutoModel.from_pretrained(MODEL_ID, revision=revision, add_pooling_layer=False)\n        self.responder = copy.deepcopy(self.seeker)\n        hidden = self.seeker.config.hidden_size\n        self.attn = CrossAttention(hidden)\n        self.dropout = nn.Dropout(dropout)\n        self.empathy_classifier = nn.Sequential(nn.Dropout(.1), nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(.1), nn.Linear(hidden, 3))\n        self.rationale_classifier = nn.Linear(hidden, 2)\n        for module in [self.attn, self.empathy_classifier, self.rationale_classifier]:\n            for sub in module.modules():\n                if isinstance(sub, nn.Linear):\n                    nn.init.normal_(sub.weight, mean=0, std=.02)\n                    nn.init.zeros_(sub.bias)\n        self.seeker.requires_grad_(False)\n        self.seeker.eval()\n\n    def train(self, mode=True):\n        super().train(mode)\n        self.seeker.eval()\n        return self\n\n    def forward(self, sp_ids, sp_mask, rp_ids, rp_mask, labels=None, rationale_labels=None, lambda_ei=.5, lambda_re=.5):\n        with torch.no_grad():\n            seeker = self.seeker(input_ids=sp_ids, attention_mask=sp_mask).last_hidden_state\n        responder = self.responder(input_ids=rp_ids, attention_mask=rp_mask).last_hidden_state\n        sequence = responder + self.dropout(self.attn(responder, seeker, seeker, sp_mask))\n        logits = self.empathy_classifier(sequence[:, 0])\n        rationale_logits = self.rationale_classifier(self.dropout(sequence))\n        result = {\'logits\': logits, \'rationale_logits\': rationale_logits}\n        if labels is not None:\n            loss_ei = nn.functional.cross_entropy(logits, labels)\n            active = rationale_labels != -100 if rationale_labels is not None else None\n            loss_re = (nn.functional.cross_entropy(rationale_logits[active], rationale_labels[active])\n                       if active is not None and active.any() else rationale_logits.sum() * 0.)\n            result.update(loss=lambda_ei * loss_ei + lambda_re * loss_re, loss_ei=loss_ei, loss_re=loss_re)\n        return result\n\n\ndef evaluate(model, loader, device):\n    model.eval()\n    logits, labels, rationale_true, rationale_pred = [], [], [], []\n    with torch.inference_mode():\n        for batch in loader:\n            batch = {key: value.to(device) for key, value in batch.items()}\n            result = model(**batch)\n            logits.extend(result[\'logits\'].float().cpu().tolist())\n            if \'labels\' in batch:\n                labels.extend(batch[\'labels\'].cpu().tolist())\n                active = batch[\'rationale_labels\'] != -100\n                if active.any():\n                    rationale_true.extend(batch[\'rationale_labels\'][active].cpu().tolist())\n                    rationale_pred.extend(result[\'rationale_logits\'].argmax(-1)[active].cpu().tolist())\n    arr = np.asarray(logits)\n    if not labels:\n        return {}, arr\n    predicted = arr.argmax(1)\n    matrix = confusion_matrix(labels, predicted, labels=[0, 1, 2])\n    metrics = {\n        \'n\': len(labels), \'accuracy\': float(accuracy_score(labels, predicted)),\n        \'macro_f1\': float(f1_score(labels, predicted, labels=[0, 1, 2], average=\'macro\', zero_division=0)),\n        \'confusion_matrix_0_1_2\': matrix.tolist(),\n        \'classification_report\': classification_report(labels, predicted, labels=[0, 1, 2], output_dict=True, zero_division=0),\n        \'classes_not_predicted\': [i for i in range(3) if i not in set(predicted)],\n    }\n    if rationale_true:\n        metrics[\'rationale_token_f1_positive\'] = float(f1_score(rationale_true, rationale_pred, zero_division=0))\n        metrics[\'n_rationale_tokens_evaluated\'] = len(rationale_true)\n    return metrics, arr\n\n\ndef load_checkpoint(model, path, device):\n    state = torch.load(path, map_location=\'cpu\', weights_only=True)\n    info = model.load_state_dict(state, strict=False)\n    if info.unexpected_keys or any(not key.startswith(\'seeker.\') for key in info.missing_keys):\n        raise ValueError(f\'Checkpoint incompatível: {info}\')\n    model.to(device)\n\n\ndef train_task(task, train_dataset, val_dataset, cfg, output, revision, device):\n    seed_everything(cfg[\'seed\'])\n    model = EpitomeXLMR(revision=revision, dropout=cfg[\'dropout\']).to(device)\n    parameters = [p for p in model.parameters() if p.requires_grad]\n    # Reduz memória de ativações. O encoder seeker já está congelado.\n    model.responder.gradient_checkpointing_enable()\n    generator = torch.Generator().manual_seed(cfg[\'seed\'])\n    loader = DataLoader(train_dataset, batch_size=cfg[\'batch_size\'], shuffle=True, generator=generator)\n    val_loader = DataLoader(val_dataset, batch_size=cfg[\'eval_batch_size\'])\n    optimizer = torch.optim.AdamW(parameters, lr=cfg[\'lr\'], eps=1e-8, weight_decay=0.)\n    n_steps = math.ceil(len(loader) / cfg[\'gradient_accumulation\']) * cfg[\'epochs\']\n    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=n_steps)\n    use_amp = device.type == \'cuda\'\n    scaler = torch.amp.GradScaler(\'cuda\', enabled=use_amp)\n    best_f1, history = -1., []\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    best_path = output / \'best_model.pt\'\n    for epoch in range(cfg[\'epochs\']):\n        model.train()\n        optimizer.zero_grad(set_to_none=True)\n        losses = []\n        bar = tqdm(loader, desc=f\'{task}: época {epoch+1}/{cfg["epochs"]}\')\n        for step, batch in enumerate(bar):\n            batch = {key: value.to(device) for key, value in batch.items()}\n            # Ajusta o último grupo incompleto para não subponderar o gradiente.\n            group_start = (step // cfg[\'gradient_accumulation\']) * cfg[\'gradient_accumulation\']\n            divisor = min(cfg[\'gradient_accumulation\'], len(loader) - group_start)\n            with torch.autocast(device_type=\'cuda\', dtype=torch.float16, enabled=use_amp):\n                result = model(**batch, lambda_ei=cfg[\'lambda_ei\'], lambda_re=cfg[\'lambda_re\'])\n                loss = result[\'loss\'] / divisor\n            if not torch.isfinite(loss):\n                raise RuntimeError(f\'{task}: perda não finita. Reduza batch_size ou desative AMP.\')\n            scaler.scale(loss).backward()\n            losses.append(float(result[\'loss\'].detach().cpu()))\n            if (step + 1) % cfg[\'gradient_accumulation\'] == 0 or step + 1 == len(loader):\n                scaler.unscale_(optimizer)\n                nn.utils.clip_grad_norm_(parameters, 1.)\n                previous_scale = scaler.get_scale()\n                scaler.step(optimizer)\n                scaler.update()\n                if scaler.get_scale() >= previous_scale:\n                    scheduler.step()\n                optimizer.zero_grad(set_to_none=True)\n            bar.set_postfix(loss=f\'{np.mean(losses):.4f}\')\n        val_metrics, _ = evaluate(model, val_loader, device)\n        history.append({\'epoch\': epoch + 1, \'train_loss\': float(np.mean(losses)), \'validation_pt\': val_metrics})\n        write_json(output / \'training_history.json\', history)\n        print(f\'{task}: validação PT Macro-F1={val_metrics["macro_f1"]:.4f}\')\n        if val_metrics[\'macro_f1\'] > best_f1:\n            best_f1 = val_metrics[\'macro_f1\']\n            # Seeker é reconstruído a partir da revisão base fixada no manifesto.\n            state = {key: value.detach().cpu() for key, value in model.state_dict().items() if not key.startswith(\'seeker.\')}\n            temporary = output / \'best_model.tmp\'\n            torch.save(state, temporary)\n            temporary.replace(best_path)\n            del state\n    del optimizer, scheduler, scaler\n    gc.collect()\n    load_checkpoint(model, best_path, device)\n    return model, history\n\n\ndef rationale_text(text, offsets, predicted):\n    spans = []\n    for (start, end), positive in zip(offsets, predicted):\n        if not positive or end <= start:\n            continue\n        if spans and start <= spans[-1][1] + 1:\n            spans[-1][1] = max(spans[-1][1], end)\n        else:\n            spans.append([start, end])\n    return \' | \'.join(text[start:end] for start, end in spans)\n\n\ndef predict_pairs(model, dataset, device, batch_size):\n    model.eval()\n    probabilities, logits, texts = [], [], []\n    position = 0\n    with torch.inference_mode():\n        for batch in DataLoader(dataset, batch_size=batch_size):\n            result = model(**{key: value.to(device) for key, value in batch.items()})\n            logits.extend(result[\'logits\'].float().cpu().tolist())\n            probabilities.extend(result[\'logits\'].float().softmax(-1).cpu().tolist())\n            rationale = result[\'rationale_logits\'].argmax(-1).cpu().tolist()\n            for labels in rationale:\n                row = dataset.frame.iloc[position]\n                texts.append(rationale_text(row.response_post, dataset.offsets[position], labels))\n                position += 1\n    return np.asarray(logits), np.asarray(probabilities), texts\n\n\ndef exact_sign_test(higher, lower):\n    n = higher + lower\n    if not n:\n        return 1.\n    return min(1., 2 * sum(math.comb(n, i) for i in range(min(higher, lower) + 1)) / 2**n)\n\n\ndef holm(pvalues):\n    result, previous = [0.] * len(pvalues), 0.\n    for rank, i in enumerate(sorted(range(len(pvalues)), key=lambda k: pvalues[k])):\n        previous = max(previous, min(1., (len(pvalues) - rank) * pvalues[i]))\n        result[i] = previous\n    return result\n\n\ndef regression_table(pairs):\n    """Análise adicional comparável à tabela ordinal, com SE por enunciado."""\n    import statsmodels.api as sm\n    from statsmodels.miscmodels.ordinal_model import OrderedModel\n    rows, status = [], {}\n    x = pd.DataFrame({model: (pairs.id == model).astype(float) for model in [\'LLAMA\', \'GPT4\']})\n    for task in TASKS:\n        y = pairs[task + \'_label\'].astype(int)\n        if y.nunique() < 2:\n            status[task] = {\'available\': False, \'reason\': \'Only one observed score level\'}\n            continue\n        # With intercept + model indicators, any group with a constant binary\n        # outcome puts its group probability at a boundary. MLE is not finite,\n        # even when the GLM optimizer reports convergence without warnings.\n        if y.nunique() == 2:\n            constant_groups = [model for model in MODELS if y[pairs.id == model].nunique() < 2]\n            if constant_groups:\n                status[task] = {\'available\': False,\n                                \'reason\': \'Complete or quasi-complete separation: constant outcome within model groups; ordinary logistic MLE is not finite\',\n                                \'constant_outcome_models\': constant_groups,\n                                \'n_clusters\': int(pairs.prompt_id.nunique())}\n                continue\n        try:\n            with warnings.catch_warnings(record=True) as caught:\n                warnings.simplefilter(\'always\')\n                if y.nunique() == 2:\n                    result = sm.GLM((y == y.max()).astype(int), sm.add_constant(x), family=sm.families.Binomial()).fit(\n                        cov_type=\'cluster\', cov_kwds={\'groups\': pairs.prompt_id, \'use_correction\': True})\n                    model_type = \'binary_logistic_two_observed_levels\'\n                    converged = bool(result.converged)\n                else:\n                    result = OrderedModel(y, x, distr=\'logit\').fit(method=\'bfgs\', disp=False, cov_type=\'cluster\',\n                        cov_kwds={\'groups\': pairs.prompt_id, \'use_correction\': True})\n                    model_type = \'proportional_odds_ordinal_logistic\'\n                    converged = bool(result.mle_retvals.get(\'converged\', False))\n            messages = [str(w.message) for w in caught]\n            fatal = any(re.search(r\'separation|singular|hessian|overflow|converg\', message, re.I) for message in messages)\n            if not converged or fatal:\n                status[task] = {\'available\': False, \'reason\': \'Fit did not produce a reliable estimate\', \'warnings\': messages}\n                continue\n            ci = result.conf_int()\n            for model in [\'LLAMA\', \'GPT4\']:\n                coef, low, high = float(result.params[model]), float(ci.loc[model, 0]), float(ci.loc[model, 1])\n                pvalue = float(result.pvalues[model])\n                if not np.isfinite([coef, low, high, pvalue]).all() or max(abs(low), abs(high)) > 50:\n                    raise ValueError(\'Unstable coefficient or confidence interval\')\n                rows.append({\'metric\': task, \'model\': model, \'reference\': \'GEMMA\', \'fit_type\': model_type,\n                             \'log_odds\': coef, \'log_odds_ci_low\': low, \'log_odds_ci_high\': high,\n                             \'odds_ratio\': math.exp(coef), \'odds_ratio_ci_low\': math.exp(low),\n                             \'odds_ratio_ci_high\': math.exp(high), \'p_cluster_wald\': pvalue})\n            status[task] = {\'available\': True, \'fit_type\': model_type, \'n_clusters\': int(pairs.prompt_id.nunique()),\n                            \'warnings\': messages, \'note\': \'Small sample: use the exact paired tests as the primary analysis.\'}\n        except Exception as exc:\n            rows = [r for r in rows if r[\'metric\'] != task]\n            status[task] = {\'available\': False, \'reason\': str(exc)}\n    # Família pré-definida: duas comparações com Gemma por dimensão (6).\n    positions = [(task, model) for task in TASKS for model in [\'LLAMA\', \'GPT4\']]\n    lookup = {(r[\'metric\'], r[\'model\']): r for r in rows}\n    adjusted = holm([lookup[p][\'p_cluster_wald\'] if p in lookup else 1. for p in positions])\n    for position, pvalue in zip(positions, adjusted):\n        if position in lookup:\n            lookup[position][\'p_cluster_wald_holm_6\'] = pvalue\n    columns = [\'metric\', \'model\', \'reference\', \'fit_type\', \'log_odds\', \'log_odds_ci_low\', \'log_odds_ci_high\',\n               \'odds_ratio\', \'odds_ratio_ci_low\', \'odds_ratio_ci_high\', \'p_cluster_wald\', \'p_cluster_wald_holm_6\']\n    return pd.DataFrame(rows, columns=columns), status\n\n\ndef analysis_tables(pairs):\n    if len(pairs) != 60 or pairs.id.value_counts().to_dict() != {model: 20 for model in MODELS}:\n        raise ValueError(\'São necessários exatamente 20 pares de cada modelo.\')\n    grouped = pairs.groupby(\'prompt_id\')\n    if grouped.ngroups != 20 or not all(set(g.id) == set(MODELS) for _, g in grouped):\n        raise ValueError(\'Os modelos não compartilham os mesmos 20 enunciados.\')\n    summary, tests = [], []\n    for task in TASKS:\n        for model in MODELS:\n            labels = pairs.loc[pairs.id == model, task + \'_label\'].astype(int)\n            if not labels.isin([0, 1, 2]).all():\n                raise ValueError(\'Rótulos fora de 0/1/2.\')\n            summary.append({\'metric\': task, \'model\': model, \'n\': len(labels),\n                            **{f\'count_{i}\': int((labels == i).sum()) for i in range(3)},\n                            \'mean_ordinal_score_descriptive\': float(labels.mean())})\n        wide = pairs.pivot(index=\'prompt_id\', columns=\'id\', values=task + \'_label\')\n        for a, b in itertools.combinations(MODELS, 2):\n            differences = wide[a].astype(int) - wide[b].astype(int)\n            higher, lower, ties = int((differences > 0).sum()), int((differences < 0).sum()), int((differences == 0).sum())\n            tests.append({\'metric\': task, \'model_A\': a, \'model_B\': b,\n                          \'A_score_higher\': higher, \'B_score_higher\': lower, \'ties\': ties,\n                          \'p_exact_paired_sign\': exact_sign_test(higher, lower)})\n    for row, adjusted in zip(tests, holm([r[\'p_exact_paired_sign\'] for r in tests])):\n        row.update(p_holm_9=adjusted, significant_at_0_05=bool(adjusted < .05))\n    return pd.DataFrame(summary), pd.DataFrame(tests)\n\n\ndef compare_baseline(pairs, baseline_path, name):\n    baseline = pd.read_csv(baseline_path).fillna(\'\')\n    def key(row):\n        return tuple(normalize(row[c]) for c in [\'id\', \'seeker_post\', \'response_post\'])\n    old = {key(row): row for _, row in baseline.iterrows()}\n    current = [key(row) for _, row in pairs.iterrows()]\n    if len(old) != 60 or len(set(current)) != 60 or set(old) != set(current):\n        raise ValueError(f\'{name}: os 60 pares não coincidem.\')\n    result = pairs[[\'prompt_id\', \'id\', \'seeker_post\', \'response_post\']].copy()\n    for task in TASKS:\n        result[task + \'_baseline\'] = [int(old[k][task + \'_label\']) for k in current]\n        result[task + \'_epitome_xlmr\'] = pairs[task + \'_label\'].astype(int).to_numpy()\n        result[task + \'_changed\'] = result[task + \'_baseline\'] != result[task + \'_epitome_xlmr\']\n    return result\n\n\ndef write_report(output, cfg, summary, tests, metrics, regression_status):\n    lines = [\'# EPITOME com XLM-R: resultados da execução\', \'\',\n             \'Modelo base: `FacebookAI/xlm-roberta-base`. Adaptação bi-encoder multitarefa, com encoder seeker congelado, cross-attention, cabeça de níveis 0/1/2 e cabeça de rationales.\', \'\',\n             \'As traduções PT recebem supervisão de nível. Rationales são supervisionadas apenas nas versões EN originais, porque há trechos incompatíveis nas traduções. A versão EN e PT de cada enunciado fica na mesma divisão. Seleção do checkpoint: Macro-F1 na validação PT. Teste EN/PT independente da seleção.\', \'\',\n             \'Esta é uma adaptação com processamento atualizado, treino bilíngue e divisão por grupos; não é uma reprodução numérica exata do treinamento anterior.\', \'\',\n             \'## Distribuição nos 60 pares\', \'\', \'| Dimensão | Modelo | Nível 0 | Nível 1 | Nível 2 | Média descritiva |\', \'|---|---|---:|---:|---:|---:|\']\n    for row in summary.itertuples():\n        lines.append(f\'| {row.metric} | {row.model} | {row.count_0} | {row.count_1} | {row.count_2} | {row.mean_ordinal_score_descriptive:.3f} |\')\n    lines += [\'\', \'## Testes pareados primários\', \'\',\n              \'Teste exato dos sinais, bilateral: compara quantas vezes um modelo recebeu nível maior ou menor no mesmo enunciado. Empates são excluídos do número informativo. Holm em nove comparações predefinidas. Esse teste usa a ordem das classes sem supor distâncias iguais. Com apenas dois níveis, coincide com McNemar exato.\', \'\',\n              \'| Dimensão | Comparação | A maior | B maior | Empates | p exato | p Holm |\', \'|---|---|---:|---:|---:|---:|---:|\']\n    for row in tests.itertuples():\n        lines.append(f\'| {row.metric} | {row.model_A} × {row.model_B} | {row.A_score_higher} | {row.B_score_higher} | {row.ties} | {row.p_exact_paired_sign:.5f} | {row.p_holm_9:.5f} |\')\n    lines += [\'\', \'## Teste em dados anotados e traduzidos\', \'\', \'| Dimensão | Idioma | N | Acurácia | Macro-F1 | Classes não previstas |\', \'|---|---|---:|---:|---:|---|\']\n    quality_flags = []\n    for task in TASKS:\n        for lang in [\'en\', \'pt\']:\n            m = metrics[task][\'test_\' + lang]\n            lines.append(f\'| {task} | {lang} | {m["n"]} | {m["accuracy"]:.4f} | {m["macro_f1"]:.4f} | {m["classes_not_predicted"]} |\')\n            if m[\'classes_not_predicted\']:\n                quality_flags.append(f\'{task}/{lang}: classes não previstas no teste: {m["classes_not_predicted"]}.\')\n    lines += [\'\', \'## Limites para o artigo\', \'\',\n              \'Os níveis das traduções PT são transferidos das anotações EN, e não foram reanotados independentemente por falantes de português. As predições nos 60 pares não demonstram eficácia clínica. A validação humana específica dos níveis ER/IP/EX é distinta de avaliações humanas de outros comportamentos de entrevista motivacional. O template de revisão humana oculta as predições para reduzir viés de anotação.\', \'\',\n              \'A regressão ordinal/binária é uma análise adicional com erro-padrão agrupado por enunciado (20 grupos). A amostra é pequena. Falhas de convergência/separação são informadas em `regression_status.json`, sem criar estimativas artificiais. Log-odds, odds ratio, IC e p-valor têm colunas próprias. Resultados de significância devem priorizar os testes pareados exatos.\', \'\']\n    lines += [\'- \' + flag for flag in quality_flags] or [\'Nenhuma classe ausente nas predições de teste.\']\n    lines += [\'\', \'Fonte da arquitetura e escala: Sharma et al. (2020), https://aclanthology.org/2020.emnlp-main.425/.\',\n              \'Detalhes de fontes, revisões e ambiente: `manifest.json`, `pip_freeze.txt` e `fontes_e_alinhamento.json`.\', \'\']\n    (output / \'relatorio_resultados.md\').write_text(\'\\n\'.join(lines), encoding=\'utf-8\')\n    write_json(output / \'quality_flags.json\', {\'classes_not_predicted\': quality_flags, \'independent_human_pt_validation\': False})\n    train_n = metrics[\'ER\'][\'n_training_examples\']\n    val_n = metrics[\'ER\'][\'validation_pt\'][\'n\']\n    test_n = metrics[\'ER\'][\'test_pt\'][\'n\']\n    manuscript = [\n        \'# Draft methods and results for the manuscript\', \'\',\n        f\'We implemented a multilingual adaptation of the EPITOME multi-task bi-encoder. Both encoders were initialized from `FacebookAI/xlm-roberta-base`. The seeker encoder was frozen and kept in evaluation mode. The responder representation was augmented by cross-attention over the seeker representation and used by a three-level classification head and a binary token-level rationale head. Padding tokens were masked in cross-attention. Special and padding tokens were excluded from rationale loss.\', \'\',\n        f\'For each mechanism (ER, IP, and EX), training used {train_n} examples comprising English annotated pairs and their Portuguese translations. Approximately 60% of the source rows were used for training, 20% for validation, and 20% for testing. Rows sharing a seeker identifier or normalized seeker text were grouped; English and Portuguese versions stayed in the same split. Rationale supervision used only original English spans that could be aligned to response characters. Missing or unmatched spans were excluded from that auxiliary loss. Portuguese examples received level supervision from transferred English labels. These translations were not independently reannotated by Portuguese-speaking raters.\', \'\',\n        f\'Training used {cfg["epochs"]} epochs, learning rate {cfg["lr"]}, effective batch size {cfg["batch_size"] * cfg["gradient_accumulation"]}, seed {cfg["seed"]}, maximum sequence length {cfg["max_length"]} per encoder, and objective L = {cfg["lambda_ei"]} L_level + {cfg["lambda_re"]} L_rationale. The checkpoint was selected by macro-F1 on {val_n} Portuguese validation examples. The corresponding {test_n}-example English and Portuguese test splits were held out from checkpoint selection. Dataset revisions, model revision, software versions, split assignments, truncation counts, and alignment exclusions were recorded in the run manifest and audit files.\', \'\',\n        \'The same 20 prompts were evaluated for each generation model. Primary between-model comparisons used two-sided exact paired sign tests on ordinal score differences, with Holm adjustment across the nine predefined comparisons (three model pairs for each of ER, IP, and EX). Ties were excluded from the informative pair count. Additional proportional-odds models, or binary logistic models when only two levels were observed, used Gemma as reference and standard errors clustered by prompt. These asymptotic estimates were treated as supplementary because there were only 20 prompt clusters.\', \'\',\n        \'## Held-out performance\', \'\',\n        \'| Mechanism | English test macro-F1 | Portuguese translated test macro-F1 |\',\n        \'|---|---:|---:|\',\n    ]\n    for task in TASKS:\n        manuscript.append(f\'| {task} | {metrics[task]["test_en"]["macro_f1"]:.4f} | {metrics[task]["test_pt"]["macro_f1"]:.4f} |\')\n    manuscript += [\'\', \'## Score distribution\', \'\', \'| Mechanism | Model | Score 0 | Score 1 | Score 2 |\', \'|---|---|---:|---:|---:|\']\n    for row in summary.itertuples():\n        manuscript.append(f\'| {row.metric} | {row.model} | {row.count_0} | {row.count_1} | {row.count_2} |\')\n    significant = tests[tests.significant_at_0_05]\n    manuscript += [\'\', \'## Primary paired comparisons\', \'\']\n    if significant.empty:\n        manuscript.append(\'None of the nine paired comparisons met the adjusted 0.05 significance threshold. This does not establish equivalence between models.\')\n    else:\n        for row in significant.itertuples():\n            manuscript.append(f\'{row.metric}: {row.model_A} versus {row.model_B}, exact p = {row.p_exact_paired_sign:.5f}, Holm-adjusted p = {row.p_holm_9:.5f}; {row.A_score_higher} prompts favored A and {row.B_score_higher} favored B, with {row.ties} ties.\')\n    manuscript += [\'\', \'## Limitations to retain in the paper\', \'\',\n        \'This is an adaptation with bilingual supervision and updated preprocessing, rather than an exact numerical reproduction of the earlier training run. Evaluation on translated annotations does not directly validate Portuguese ER/IP/EX scores; human assessment of other motivational interviewing behaviors is complementary. Predicted empathy mechanisms and model-to-model differences do not establish clinical effectiveness. Softmax probabilities were not calibrated. Missing predicted classes and alignment/truncation exclusions should be reported alongside overall performance.\', \'\',\n        \'Framework source: Sharma et al. (2020), https://aclanthology.org/2020.emnlp-main.425/.\', \'\']\n    (output / \'draft_methods_results_EN.md\').write_text(\'\\n\'.join(manuscript), encoding=\'utf-8\')\n\n\ndef run_experiment(cfg, base_output, original_pairs_path, additional_baseline_path, source_dir=\'/content/epitome_sources\'):\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'Ative GPU no Colab: Ambiente de execução > Alterar tipo de ambiente de execução > GPU.\')\n    if cfg[\'train_languages\'] != [\'en\', \'pt\']:\n        raise ValueError(\'Esta versão foi preparada para treino bilíngue EN/PT.\')\n    seed_everything(cfg[\'seed\'])\n    device = torch.device(\'cuda\')\n    hf_config = AutoConfig.from_pretrained(MODEL_ID)\n    revision = hf_config._commit_hash\n    if hf_config.model_type != \'xlm-roberta\' or not revision:\n        raise ValueError(\'Backbone ou revisão não resolvido com segurança.\')\n    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=revision, use_fast=True)\n    if not tokenizer.is_fast:\n        raise ValueError(\'É necessário tokenizer rápido com offsets de caracteres.\')\n    frames, source_audit = download_sources(source_dir)\n    split, groups = grouped_split(frames, cfg[\'seed\'])\n    code_hash = hashlib.sha256(Path(__file__).read_bytes()).hexdigest()\n    pairs = pd.read_csv(original_pairs_path).fillna(\'\')\n    baseline_hash = hashlib.sha256(Path(original_pairs_path).read_bytes()).hexdigest()\n    identity = {\'config\': cfg, \'model_id\': MODEL_ID, \'model_revision\': revision,\n                \'sources\': BLOBS, \'code_sha256\': code_hash, \'baseline_sha256\': baseline_hash,\n                \'additional_baseline_sha256\': hashlib.sha256(Path(additional_baseline_path).read_bytes()).hexdigest(),\n                \'runtime_versions\': {\'torch\': torch.__version__, \'transformers\': transformers.__version__,\n                                     \'numpy\': np.__version__, \'pandas\': pd.__version__, \'sklearn\': sklearn.__version__}}\n    run_hash = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:12]\n    output = Path(base_output) / (\'run_\' + run_hash)\n    output.mkdir(parents=True, exist_ok=True)\n    pairs = pairs[[\'id\', \'seeker_post\', \'response_post\']].copy()\n    prompt_ids = {text: i + 1 for i, text in enumerate(dict.fromkeys(pairs.seeker_post.map(normalize)))}\n    pairs.insert(0, \'prompt_id\', pairs.seeker_post.map(lambda text: prompt_ids[normalize(text)]))\n    # Valida os pares antes de gastar tempo de GPU.\n    dummy = pairs.assign(ER_label=0, IP_label=0, EX_label=0)\n    analysis_tables(dummy)\n    compare_baseline(dummy, original_pairs_path, \'original\')\n    compare_baseline(dummy, additional_baseline_path, \'additional\')\n    manifest = dict(identity, run_id=run_hash, status=\'running\',\n                    selection=\'best validation PT macro-F1; untouched held-out EN/PT test\',\n                    train_count_per_language=len(split[\'train\']), validation_count_per_language=len(split[\'validation\']),\n                    test_count_per_language=len(split[\'test\']), frozen_seeker=True, saved_checkpoint_excludes_frozen_seeker=True,\n                    architecture_adaptations=[\'Modern XLM-R encoders\', \'Character-offset rationale alignment\',\n                        \'Ignore missing/unmatched rationale spans\', \'No supervision on special/padding tokens\',\n                        \'Mask padding in cross-attention\', \'Frozen seeker in eval mode\',\n                        \'Bilingual level supervision; rationale supervision only on EN\', \'Group-disjoint split\'],\n                    torch=torch.__version__, transformers=transformers.__version__, numpy=np.__version__,\n                    pandas=pd.__version__, sklearn=sklearn.__version__, gpu=torch.cuda.get_device_name(0))\n    write_json(output / \'manifest.json\', manifest)\n    write_json(output / \'fontes_e_alinhamento.json\', source_audit)\n    split_frame = pd.DataFrame({\'row_id\': np.arange(len(groups)), \'group_id\': groups, \'split\': \'\'})\n    for name, ids in split.items():\n        split_frame.loc[ids, \'split\'] = name\n    split_frame.to_csv(output / \'split_assignments.csv\', index=False)\n    freeze = subprocess.run([\'python\', \'-m\', \'pip\', \'freeze\'], capture_output=True, text=True, check=True)\n    (output / \'pip_freeze.txt\').write_text(freeze.stdout, encoding=\'utf-8\')\n    shutil.copyfile(__file__, output / \'epitome_xlmr.py\')\n    shutil.copyfile(original_pairs_path, output / \'baseline_original.csv\')\n    shutil.copyfile(additional_baseline_path, output / \'baseline_classificador_pares_pt.csv\')\n    tokenizer.save_pretrained(output / \'tokenizer\')\n    metrics, preprocessing = {}, {}\n    for task in TASKS:\n        print(f\'\\n===== {task}; resultados em {output} =====\')\n        task_output = output / task\n        task_output.mkdir(exist_ok=True)\n        datasets = {}\n        for lang in [\'en\', \'pt\']:\n            for name, ids in split.items():\n                datasets[lang + \'_\' + name] = EncodedPairs(frames[lang][task].iloc[ids], tokenizer, cfg[\'max_length\'], supervise_rationales=(lang == \'en\'))\n        preprocessing[task] = {name: dict(ds.audit) for name, ds in datasets.items()}\n        write_json(output / \'preprocessing_audit.json\', preprocessing)\n        completed = task_output / \'completed.json\'\n        if completed.exists() and (task_output / \'best_model.pt\').exists():\n            print(f\'{task}: checkpoint e resultados concluídos recuperados do Drive.\')\n            model = EpitomeXLMR(revision=revision, dropout=cfg[\'dropout\'])\n            load_checkpoint(model, task_output / \'best_model.pt\', device)\n            history = json.loads((task_output / \'training_history.json\').read_text())\n        else:\n            training = torch.utils.data.ConcatDataset([datasets[\'en_train\'], datasets[\'pt_train\']])\n            model, history = train_task(task, training, datasets[\'pt_validation\'], cfg, task_output, revision, device)\n        task_metrics = {\'best_epoch\': max(history, key=lambda entry: entry[\'validation_pt\'][\'macro_f1\'])[\'epoch\'],\n                        \'n_training_examples\': len(datasets[\'en_train\']) + len(datasets[\'pt_train\'])}\n        for lang in [\'en\', \'pt\']:\n            for name in [\'validation\', \'test\']:\n                ds = datasets[lang + \'_\' + name]\n                m, logits = evaluate(model, DataLoader(ds, batch_size=cfg[\'eval_batch_size\']), device)\n                task_metrics[name + \'_\' + lang] = m\n                pred = ds.frame[[\'row_id\', \'level\']].copy()\n                pred[\'predicted_level\'] = logits.argmax(1)\n                for i in range(3):\n                    pred[f\'logit_{i}\'] = logits[:, i]\n                pred.to_csv(task_output / f\'{name}_{lang}_predictions.csv\', index=False)\n        metrics[task] = task_metrics\n        write_json(output / \'validation_and_test.json\', metrics)\n        pair_dataset = EncodedPairs(pairs[[\'id\', \'seeker_post\', \'response_post\']], tokenizer, cfg[\'max_length\'])\n        logits, probabilities, rationales = predict_pairs(model, pair_dataset, device, cfg[\'eval_batch_size\'])\n        pairs[task + \'_label\'] = logits.argmax(1)\n        for i in range(3):\n            pairs[f\'{task}_logit_{i}\'] = logits[:, i]\n            pairs[f\'{task}_probability_{i}\'] = probabilities[:, i]\n        pairs[task + \'_rationale_text\'] = rationales\n        pairs.to_csv(output / \'rotulos_parciais.csv\', index=False)\n        write_json(task_output / \'completed.json\', {\'task\': task, \'run_id\': run_hash, \'best_epoch\': task_metrics[\'best_epoch\']})\n        del model, datasets, pair_dataset\n        gc.collect()\n        torch.cuda.empty_cache()\n    pairs.to_csv(output / \'rotulos_epitome_xlmr_60_pares.csv\', index=False)\n    summary, tests = analysis_tables(pairs)\n    summary.to_csv(output / \'resumo_por_modelo.csv\', index=False)\n    tests.to_csv(output / \'testes_pareados.csv\', index=False)\n    regression, regression_status = regression_table(pairs)\n    regression.to_csv(output / \'regressao_log_odds_e_OR.csv\', index=False)\n    write_json(output / \'regression_status.json\', regression_status)\n    for path, name in [(original_pairs_path, \'original\'), (additional_baseline_path, \'classificador_pares_pt\')]:\n        comparison = compare_baseline(pairs, path, name)\n        comparison.to_csv(output / f\'comparacao_{name}.csv\', index=False)\n    review = pairs[[\'prompt_id\', \'id\', \'seeker_post\', \'response_post\']].copy()\n    for task in TASKS:\n        for rater in [\'rater_1\', \'rater_2\', \'adjudicated\']:\n            review[f\'{task}_{rater}\'] = \'\'\n    review.to_csv(output / \'template_validacao_humana_pt.csv\', index=False)\n    write_report(output, cfg, summary, tests, metrics, regression_status)\n    manifest[\'status\'] = \'completed\'\n    write_json(output / \'manifest.json\', manifest)\n    # Tabelas em LaTeX para o manuscrito (método/limites descritos no relatório).\n    summary.to_latex(output / \'tabela_distribuicao.tex\', index=False, escape=True, float_format=\'%.4f\')\n    tests.to_latex(output / \'tabela_testes_pareados.tex\', index=False, escape=True, float_format=\'%.5f\')\n    if not regression.empty:\n        regression[[\'metric\', \'model\', \'odds_ratio\', \'odds_ratio_ci_low\', \'odds_ratio_ci_high\', \'p_cluster_wald\', \'p_cluster_wald_holm_6\']].to_latex(\n            output / \'tabela_regressao_OR.tex\', index=False, escape=True, float_format=\'%.5f\')\n    import zipfile\n    zip_path = output / \'resultados_epitome_xlmr.zip\'\n    with zipfile.ZipFile(zip_path, \'w\', compression=zipfile.ZIP_DEFLATED) as archive:\n        for path in output.rglob(\'*\'):\n            if path.is_file() and path.suffix not in {\'.pt\', \'.tmp\', \'.zip\'}:\n                archive.write(path, arcname=str(path.relative_to(output)))\n    print(f\'\\nConcluído. ZIP de resultados: {zip_path}\')\n    print(\'Os três checkpoints permanecem no Drive, fora do ZIP de tabelas.\')\n    return output, zip_path, summary, tests\n', encoding='utf-8')
# Entradas públicas da execução usada no artigo. Defina False para enviar outras.
USE_PUBLISHED_BASELINES = True
required = ['baseline_original.csv', 'baseline_classificador_pares_pt.csv']
if USE_PUBLISHED_BASELINES:
    import requests, hashlib, json
    base = 'https://raw.githubusercontent.com/JPSchettino/model_analisys-/main/results/2026-10-01/'
    for name, expected in [(required[0], 'ca91bf35da243ce0ee74f200724efb4c882a2fc78605b940100dccf2970c6f3b'),
                           (required[1], '44f971eb6d34c4ea85de2e2754f2374c899987defda6411e99684ed1371d98ae')]:
        response = requests.get(base + name, timeout=60)
        response.raise_for_status()
        if hashlib.sha256(response.content).hexdigest() != expected:
            raise ValueError('Hash inesperado: ' + name)
        Path('/content', name).write_bytes(response.content)
else:
    from google.colab import files
    uploaded = files.upload()
    missing = [name for name in required if name not in uploaded]
    if missing:
        raise ValueError('Faltam os arquivos: ' + ', '.join(missing))
    for name in required:
        Path('/content', name).write_bytes(uploaded[name])
print('Código e arquivos de avaliação preparados.')


In [ ]:
import importlib, sys, transformers
from packaging.version import Version
if transformers.__version__ != '4.48.3':
    raise RuntimeError('Reinicie a sessão do Colab para carregar transformers 4.48.3 e execute tudo novamente.')
import torch
sys.path.insert(0, '/content')
import epitome_xlmr as e
e = importlib.reload(e)
assert e.MODEL_ID == 'FacebookAI/xlm-roberta-base'
assert Version(torch.__version__.split('+')[0]) >= Version('2.3')
if not torch.cuda.is_available():
    raise RuntimeError('Selecione GPU em Ambiente de execução > Alterar tipo de ambiente de execução.')
print('GPU:', torch.cuda.get_device_name(0))
print('Memória total (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

from google.colab import drive
drive.mount('/content/drive')
BASE_OUTPUT = Path('/content/drive/MyDrive/EPITOME_XLMR_JP')
BASE_OUTPUT.mkdir(parents=True, exist_ok=True)

CONFIG = {
    'seed': 12,
    'max_length': 64,             # limite usado no código original; truncamentos são auditados
    'epochs': 4,
    'lr': 2e-5,
    'dropout': 0.1,
    'batch_size': 8,              # microbatch; adequado para T4
    'gradient_accumulation': 4,   # batch efetivo = 32
    'eval_batch_size': 8,
    'lambda_ei': 0.5,
    'lambda_re': 0.5,
    'train_languages': ['en', 'pt'],
}
print('Configuração:', CONFIG)
print('Pasta permanente:', BASE_OUTPUT)


In [ ]:
# Checagem pequena, sem baixar o modelo base. O treino completo vem na próxima célula.
from transformers import XLMRobertaConfig
tiny_config = XLMRobertaConfig(
    vocab_size=32, hidden_size=16, num_hidden_layers=1, num_attention_heads=2,
    intermediate_size=32, max_position_embeddings=32,
    pad_token_id=1, bos_token_id=0, eos_token_id=2,
)
tiny = e.EpitomeXLMR(config=tiny_config).train()
ids = torch.tensor([[0, 4, 5, 2, 1, 1], [0, 6, 7, 2, 1, 1]])
mask = (ids != 1).long()
rationales = torch.tensor([[-100, 1, 0, -100, -100, -100], [-100, 0, 1, -100, -100, -100]])
checked = tiny(ids, mask, ids, mask, labels=torch.tensor([0, 2]), rationale_labels=rationales)
assert checked['logits'].shape == (2, 3)
assert checked['rationale_logits'].shape == (2, 6, 2)
assert torch.isfinite(checked['loss'])
checked['loss'].backward()
assert all(parameter.grad is None for parameter in tiny.seeker.parameters())
assert any(parameter.grad is not None for parameter in tiny.responder.parameters())
del tiny, checked
print('Checagem de arquitetura, perdas e gradientes concluída.')


## Treinar, avaliar e gerar as análises

Execute a próxima célula e aguarde as três tarefas. A pasta de saída é identificada pelo código, configuração, fontes e revisão do modelo. Alterações criam outra pasta; tarefas concluídas da mesma execução são reutilizadas.


In [ ]:
OUTPUT, ZIP_RESULTS, SUMMARY, PAIRED_TESTS = e.run_experiment(
    CONFIG, BASE_OUTPUT,
    '/content/baseline_original.csv',
    '/content/baseline_classificador_pares_pt.csv',
)
display(SUMMARY)
display(PAIRED_TESTS)
print('Relatório:', OUTPUT / 'relatorio_resultados.md')


In [ ]:
from google.colab import files
print('Resultados também salvos no seu Drive:', OUTPUT)
print('O ZIP contém tabelas, predições, testes, logs, código e manifesto; os checkpoints ficam fora do ZIP.')
files.download(str(ZIP_RESULTS))


## Arquivos que serão gerados

- `rotulos_epitome_xlmr_60_pares.csv`: níveis, logits, probabilidades e trechos previstos de rationale.
- `resumo_por_modelo.csv`: contagem de cada nível por modelo.
- `testes_pareados.csv`: teste exato dos sinais com Holm nas nove comparações predefinidas. Preserva a ordem dos níveis 0/1/2. Se só houver dois níveis, equivale a McNemar exato.
- `regressao_log_odds_e_OR.csv`: análise adicional ordinal ou binária, referência Gemma, erros agrupados por enunciado. Falhas de estimação são informadas em `regression_status.json`. Há apenas 20 grupos; os testes exatos são a análise principal.
- `comparacao_original.csv` e `comparacao_classificador_pares_pt.csv`: alterações por par em relação às duas rodadas anteriores.
- `validation_and_test.json`: desempenho por classe, matrizes de confusão e avaliação de rationales nas versões EN.
- `preprocessing_audit.json`: trechos sem alinhamento e textos truncados.
- `template_validacao_humana_pt.csv`: 60 pares sem exibir as previsões, para dois anotadores e adjudicação.
- `relatorio_resultados.md`, tabelas `.tex`, `manifest.json` e `pip_freeze.txt`: material para analisar e descrever o experimento.
- `draft_methods_results_EN.md`: rascunho em inglês de métodos, resultados e limites, preenchido com os dados da nova execução.

Antes de atualizar as conclusões do artigo, confira o Macro-F1 de teste, as classes não previstas e o relatório. Probabilidades de softmax não são medidas de eficácia clínica nem foram calibradas neste notebook.
